In [ ]:
# Cell 1: 切到项目根目录，并把 src 加入导入路径
import os
import sys

PROJECT_ROOT = os.path.abspath("..")  # 仓库根目录（本笔记本在 notebooks/ 下）
os.chdir(PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

print("cwd =", os.getcwd())
print("src in sys.path =", os.path.join(PROJECT_ROOT, "src") in sys.path)

In [ ]:
# Cell 2: 导入训练函数与基础依赖，设置随机种子
import json
import numpy as np
import torch

from train import train  # src/train.py 里的 train()

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

USE_GPU = torch.cuda.is_available()
if USE_GPU:
    torch.cuda.manual_seed_all(SEED)

print("USE_GPU =", USE_GPU)
print("PyTorch =", torch.__version__)

In [ ]:
# Cell 3: 训练 abnormal（必须先训练它，后续 acl/meniscus 要用到 abnormal_model_path）
RUNDIR = "runs/abnormal_triple_alex"
TASK = "abnormal"
BACKBONE = "alexnet"
EPOCHS = 50
LEARNING_RATE = 1e-5

os.makedirs(RUNDIR, exist_ok=True)

args = dict(
    rundir=RUNDIR,
    task=TASK,
    seed=SEED,
    gpu=USE_GPU,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    epochs=EPOCHS,
    max_patience=5,
    factor=0.3,
    backbone=BACKBONE,
    abnormal_model=None,
)
with open(os.path.join(RUNDIR, "args.json"), "w") as f:
    json.dump(args, f, indent=4)

train(
    RUNDIR,
    TASK,
    BACKBONE,
    EPOCHS,
    LEARNING_RATE,
    USE_GPU,
    abnormal_model_path=None,
)

In [ ]:
# 计算 runs/abnormal_triple_alex 中所有模型在 valid 集上的 AUC，并打印出来

import os
import re
from pathlib import Path

import torch
from evaluate import run_model
from loader import load_data
from model import TripleMRNet

# ----------------- 解析模型文件名 -----------------
abnormal_rundir = Path("runs/abnormal_triple_alex")
pattern = re.compile(r"^val([0-9]+\.[0-9]+)_train([0-9]+\.[0-9]+)_epoch([0-9]+)$")

models = []
for fname in os.listdir(abnormal_rundir):
    m = pattern.match(fname)
    if not m:
        continue
    val_loss_in_name = float(m.group(1))
    train_loss_in_name = float(m.group(2))
    epoch = int(m.group(3))
    models.append({
        "name": fname,
        "path": str(abnormal_rundir / fname),
        "val_loss_in_name": val_loss_in_name,
        "train_loss_in_name": train_loss_in_name,
        "epoch": epoch,
    })

models_sorted = sorted(models, key=lambda x: (x["val_loss_in_name"], -x["epoch"]))

print("发现的 abnormal 模型文件：", len(models_sorted))

# ----------------- 准备验证集 -----------------
use_gpu = torch.cuda.is_available()
task = "abnormal"
train_loader, valid_loader = load_data(task, use_gpu)


def compute_binary_metrics(pred_scores, true_labels, threshold=0.5):
    pred_tensor = torch.as_tensor(pred_scores).view(-1).float()
    label_tensor = torch.as_tensor(true_labels).view(-1).float()

    pred_labels = (pred_tensor >= threshold).to(torch.int64)
    true_binary_labels = (label_tensor >= 0.5).to(torch.int64)

    tp = int(((pred_labels == 1) & (true_binary_labels == 1)).sum().item())
    tn = int(((pred_labels == 0) & (true_binary_labels == 0)).sum().item())
    fp = int(((pred_labels == 1) & (true_binary_labels == 0)).sum().item())
    fn = int(((pred_labels == 0) & (true_binary_labels == 1)).sum().item())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    total_count = tp + tn + fp + fn
    accuracy = (tp + tn) / total_count if total_count > 0 else 0.0

    return recall, f1_score, accuracy


# ----------------- 逐个模型计算评估指标 -----------------
results = []
for m in models_sorted:
    model = TripleMRNet(backbone="alexnet")
    state_dict = torch.load(m["path"], map_location=(None if use_gpu else "cpu"))
    model.load_state_dict(state_dict)
    if use_gpu:
        model = model.cuda()

    val_loss, val_auc, val_preds, val_labels = run_model(
        model, valid_loader, train=False, optimizer=None, abnormal_model_path=None
    )
    val_recall, val_f1, val_accuracy = compute_binary_metrics(val_preds, val_labels)

    results.append({
        "name": m["name"],
        "path": m["path"],
        "val_loss": float(val_loss),
        "val_auc": float(val_auc),
        "val_recall": float(val_recall),
        "val_f1": float(val_f1),
        "val_accuracy": float(val_accuracy),
        "epoch": m["epoch"],
    })
    print(
        f"{m['name']}\t val_loss={val_loss:.4f}\t val_AUC={val_auc:.4f}\t "
        f"val_recall={val_recall:.4f}\t val_f1={val_f1:.4f}\t val_acc={val_accuracy:.4f}"
    )

# ----------------- 按 AUC 从高到低排序，方便你手动挑选 -----------------
results_sorted_by_auc = sorted(results, key=lambda x: (-x["val_auc"], x["val_loss"]))

print("\n按 AUC 从高到低排序（前若干个）：")
for r in results_sorted_by_auc[:10]:
    print(
        f"{r['name']}\t AUC={r['val_auc']:.4f}\t Recall={r['val_recall']:.4f}\t "
        f"F1={r['val_f1']:.4f}\t Acc={r['val_accuracy']:.4f}\t val_loss={r['val_loss']:.4f}\t epoch={r['epoch']}"
    )

In [ ]:
BEST_ABNORMAL_MODEL_PATH = "runs/abnormal_triple_alex/val0.0996_train0.0749_epoch22"

In [ ]:
# ----------------- 用 abnormal 模型训练 ACL -----------------
import os
import json

ABNORMAL_MODEL_PATH = "runs/abnormal_triple_alex/val0.0996_train0.0749_epoch22"

RUNDIR = "runs/acl_triple_alex"
TASK = "acl"
BACKBONE = "alexnet"
EPOCHS = 50
LEARNING_RATE = 1e-5

os.makedirs(RUNDIR, exist_ok=True)

args = dict(
    rundir=RUNDIR,
    task=TASK,
    seed=SEED,
    gpu=USE_GPU,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    epochs=EPOCHS,
    max_patience=5,
    factor=0.3,
    backbone=BACKBONE,
    abnormal_model=ABNORMAL_MODEL_PATH,
)
with open(os.path.join(RUNDIR, "args.json"), "w") as f:
    json.dump(args, f, indent=4)

train(
    RUNDIR,
    TASK,
    BACKBONE,
    EPOCHS,
    LEARNING_RATE,
    USE_GPU,
    abnormal_model_path=ABNORMAL_MODEL_PATH,
)

In [ ]:
# ----------------- 用 abnormal 模型训练 meniscus -----------------

ABNORMAL_MODEL_PATH = "runs/abnormal_triple_alex/val0.0996_train0.0749_epoch22"

RUNDIR = "runs/meniscus_triple_alex"
TASK = "meniscus"
BACKBONE = "alexnet"
EPOCHS = 50
LEARNING_RATE = 1e-5

os.makedirs(RUNDIR, exist_ok=True)

args = dict(
    rundir=RUNDIR,
    task=TASK,
    seed=SEED,
    gpu=USE_GPU,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    epochs=EPOCHS,
    max_patience=5,
    factor=0.3,
    backbone=BACKBONE,
    abnormal_model=ABNORMAL_MODEL_PATH,
)
with open(os.path.join(RUNDIR, "args.json"), "w") as f:
    json.dump(args, f, indent=4)

train(
    RUNDIR,
    TASK,
    BACKBONE,
    EPOCHS,
    LEARNING_RATE,
    USE_GPU,
    abnormal_model_path=ABNORMAL_MODEL_PATH,
)

In [ ]:
# 在 Notebook 中运行
!pip install requests

In [ ]:
# 一次性评估 abnormal / acl / meniscus 三个任务下的所有 checkpoint
# 并按 AUC(降序) + val_loss(升序) 选出每个任务最佳模型路径

import os
import re
from pathlib import Path

import torch
from evaluate import run_model
from loader import load_data
from model import TripleMRNet


def compute_binary_metrics(pred_scores, true_labels, threshold=0.5):
    pred_tensor = torch.as_tensor(pred_scores).view(-1).float()
    label_tensor = torch.as_tensor(true_labels).view(-1).float()

    pred_labels = (pred_tensor >= threshold).to(torch.int64)
    true_labels_binary = (label_tensor >= 0.5).to(torch.int64)

    tp = int(((pred_labels == 1) & (true_labels_binary == 1)).sum().item())
    tn = int(((pred_labels == 0) & (true_labels_binary == 0)).sum().item())
    fp = int(((pred_labels == 1) & (true_labels_binary == 0)).sum().item())
    fn = int(((pred_labels == 0) & (true_labels_binary == 1)).sum().item())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total > 0 else 0.0
    return recall, f1_score, accuracy


def collect_checkpoints(task_name):
    run_dir = Path(f"runs/{task_name}_triple_alex")
    if not run_dir.exists():
        raise FileNotFoundError(f"目录不存在: {run_dir}")

    pattern = re.compile(r"^val([0-9]+\.[0-9]+)_train([0-9]+\.[0-9]+)_epoch([0-9]+)$")
    checkpoints = []
    for filename in os.listdir(run_dir):
        matched = pattern.match(filename)
        if matched is None:
            continue
        checkpoints.append({
            "name": filename,
            "path": str(run_dir / filename),
            "val_loss_in_name": float(matched.group(1)),
            "train_loss_in_name": float(matched.group(2)),
            "epoch": int(matched.group(3)),
        })

    if not checkpoints:
        raise ValueError(f"未发现可解析的 checkpoint: {run_dir}")

    checkpoints.sort(key=lambda item: (item["val_loss_in_name"], -item["epoch"]))
    return checkpoints


def evaluate_task_checkpoints(task_name, use_gpu):
    _, valid_loader = load_data(task_name, use_gpu)
    checkpoint_list = collect_checkpoints(task_name)
    evaluation_results = []

    print(f"\n==================== {task_name} ====================")
    print(f"发现 checkpoint 数量: {len(checkpoint_list)}")

    for checkpoint_info in checkpoint_list:
        model = TripleMRNet(backbone="alexnet")
        state_dict = torch.load(
            checkpoint_info["path"],
            map_location=(None if use_gpu else "cpu")
        )
        model.load_state_dict(state_dict)
        if use_gpu:
            model = model.cuda()

        val_loss, val_auc, val_preds, val_labels = run_model(
            model, valid_loader, train=False, optimizer=None, abnormal_model_path=None
        )
        val_recall, val_f1, val_accuracy = compute_binary_metrics(val_preds, val_labels)

        record = {
            "task": task_name,
            "name": checkpoint_info["name"],
            "path": checkpoint_info["path"],
            "epoch": checkpoint_info["epoch"],
            "val_loss": float(val_loss),
            "val_auc": float(val_auc),
            "val_recall": float(val_recall),
            "val_f1": float(val_f1),
            "val_accuracy": float(val_accuracy),
        }
        evaluation_results.append(record)

        print(
            f"{record['name']}\t"
            f"AUC={record['val_auc']:.4f}\t"
            f"Recall={record['val_recall']:.4f}\t"
            f"F1={record['val_f1']:.4f}\t"
            f"Acc={record['val_accuracy']:.4f}\t"
            f"val_loss={record['val_loss']:.4f}"
        )

    ranking = sorted(evaluation_results, key=lambda item: (-item["val_auc"], item["val_loss"]))
    best_checkpoint = ranking[0]

    print(f"\n[{task_name}] Top 5（按 AUC↓, val_loss↑）:")
    for item in ranking[:5]:
        print(
            f"{item['name']}\t"
            f"AUC={item['val_auc']:.4f}\t"
            f"Recall={item['val_recall']:.4f}\t"
            f"F1={item['val_f1']:.4f}\t"
            f"Acc={item['val_accuracy']:.4f}\t"
            f"val_loss={item['val_loss']:.4f}\t"
            f"epoch={item['epoch']}"
        )

    print(f"\n[{task_name}] 最佳 checkpoint: {best_checkpoint['path']}")
    return best_checkpoint, ranking


use_gpu = torch.cuda.is_available()
all_best_paths = {}
all_rankings = {}

for task in ["abnormal", "acl", "meniscus"]:
    best_record, ranking_records = evaluate_task_checkpoints(task, use_gpu)
    all_best_paths[task] = best_record["path"]
    all_rankings[task] = ranking_records

print("\n==================== 最终推荐路径 ====================")
print(f"DEFAULT_ABNORMAL_MODEL_PATH = \"{all_best_paths['abnormal']}\"")
print(f"DEFAULT_ACL_MODEL_PATH = \"{all_best_paths['acl']}\"")
print(f"DEFAULT_MENISCUS_MODEL_PATH = \"{all_best_paths['meniscus']}\"")

In [ ]:
import os, sys
import torch

# 保证能导入 src 下的模块
PROJECT_ROOT = os.path.abspath("..")  # 仓库根目录（本笔记本在 notebooks/ 下）
os.chdir(PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

from evaluate import run_model
from loader import load_data
from model import TripleMRNet

# 1. 指定要评估的模型路径（你训练保存的那个）
MODEL_PATH = "runs/abnormal_triple_alex/val0.0996_train0.0749_epoch22"  # 换成你自己的

# 2. 准备数据与模型
use_gpu = torch.cuda.is_available()
task = "abnormal"
train_loader, valid_loader = load_data(task, use_gpu)

model = TripleMRNet(backbone="alexnet")
state_dict = torch.load(MODEL_PATH, map_location=(None if use_gpu else "cpu"))
model.load_state_dict(state_dict)
if use_gpu:
    model = model.cuda()

# 3. 分别在 train / valid 上跑一遍，打印性能
for split_name, loader in [("train", train_loader), ("valid", valid_loader)]:
    loss, auc, preds, labels = run_model(
        model, loader, train=False, optimizer=None, abnormal_model_path=None
    )
    print(f"{split_name} loss: {loss:.4f}, {split_name} AUC: {auc:.4f}")

In [ ]:
import json
import os
import requests

# 从环境变量读取 API key，不要把真实 key 写进代码
LLM_API_KEY = os.environ.get("DASHSCOPE_API_KEY", "")
LLM_API_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"

if not LLM_API_KEY:
    raise ValueError("请先设置环境变量 DASHSCOPE_API_KEY")


def _extract_message_content(api_result):
    choices = api_result.get("choices", [])
    if not choices:
        return ""
    message = choices[0].get("message", {})
    return message.get("content", "")


def _default_structured_report(study_id, probabilities, error_message):
    return {
        "study_id": study_id,
        "summary": "LLM调用失败，返回规则兜底报告。",
        "risk_level": "medium",
        "findings": [
            f"abnormal={probabilities['abnormal']:.4f}",
            f"acl={probabilities['acl']:.4f}",
            f"meniscus={probabilities['meniscus']:.4f}",
        ],
        "recommendations": [
            "Please make the final judgment in combination with the radiologist's reading result.",
        ],
        "disclaimer": "This report is generated by AI and is for reference in scientific research and teaching only. It is not to be used for final clinical diagnosis.",
        "error": str(error_message),
    }


def explain_diagnosis_with_qwen(study_id, abnormal_prob, acl_prob, meniscus_prob, model="qwen3-max"):
    probabilities = {
        "abnormal": float(abnormal_prob),
        "acl": float(acl_prob),
        "meniscus": float(meniscus_prob),
    }

    system_prompt = (
        "你是膝关节MRI辅助报告助手。"
        "必须严格返回JSON对象，不要输出任何JSON之外的文本。"
        "文本语言应当严格是英语"
        "JSON字段固定为：study_id,summary,risk_level,findings,recommendations,disclaimer。"
        "risk_level只能是low/medium/high。findings和recommendations必须是字符串数组。"
    )

    user_prompt = {
        "study_id": study_id,
        "task": "knee_mri_structured_report",
        "probabilities": probabilities,
        "constraints": [
            "不得输出非JSON文本",
            "不得夸大结论，需体现AI辅助属性",
            "文本语言应当严格是英语",
            "必须保留disclaimer字段",
        ],
    }

    url = f"{LLM_API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {LLM_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(user_prompt, ensure_ascii=False)},
        ],
        "temperature": 0.2,
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        result = response.json()

        raw_content = _extract_message_content(result)
        structured = json.loads(raw_content)

        required_fields = ["study_id", "summary", "risk_level", "findings", "recommendations", "disclaimer"]
        for field_name in required_fields:
            if field_name not in structured:
                raise ValueError(f"缺少字段: {field_name}")

        if structured["risk_level"] not in {"low", "medium", "high"}:
            raise ValueError("risk_level 不在允许集合内")

        if not isinstance(structured["findings"], list) or not isinstance(structured["recommendations"], list):
            raise ValueError("findings/recommendations 必须为数组")

        return structured
    except Exception as exception_object:
        return _default_structured_report(study_id, probabilities, str(exception_object))

In [ ]:
import os, sys
import torch

# 保证能导入 src 下的模块
PROJECT_ROOT = os.path.abspath("..")  # 仓库根目录（本笔记本在 notebooks/ 下）
os.chdir(PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

from evaluate import run_model
from loader import load_data
from model import TripleMRNet

# 1. 指定要评估的模型路径（你训练保存的那个）
MODEL_PATH = "runs/abnormal_triple_alex/val0.0989_train0.0741_epoch23"  # 换成你自己的

# 2. 准备数据与模型
use_gpu = torch.cuda.is_available()
task = "abnormal"
train_loader, valid_loader = load_data(task, use_gpu)

model = TripleMRNet(backbone="alexnet")
state_dict = torch.load(MODEL_PATH, map_location=(None if use_gpu else "cpu"))
model.load_state_dict(state_dict)
if use_gpu:
    model = model.cuda()

def compute_binary_metrics(pred_scores, true_labels, threshold=0.5):
    pred_tensor = torch.as_tensor(pred_scores).view(-1).float()
    label_tensor = torch.as_tensor(true_labels).view(-1).float()

    pred_labels = (pred_tensor >= threshold).to(torch.int64)
    true_binary_labels = (label_tensor >= 0.5).to(torch.int64)

    tp = int(((pred_labels == 1) & (true_binary_labels == 1)).sum().item())
    tn = int(((pred_labels == 0) & (true_binary_labels == 0)).sum().item())
    fp = int(((pred_labels == 1) & (true_binary_labels == 0)).sum().item())
    fn = int(((pred_labels == 0) & (true_binary_labels == 1)).sum().item())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    total_count = tp + tn + fp + fn
    accuracy = (tp + tn) / total_count if total_count > 0 else 0.0

    return recall, f1_score, accuracy


# 3. 分别在 train / valid 上跑一遍，打印性能
for split_name, loader in [("train", train_loader), ("valid", valid_loader)]:
    loss, auc, preds, labels = run_model(
        model, loader, train=False, optimizer=None, abnormal_model_path=None
    )
    recall, f1_score, accuracy = compute_binary_metrics(preds, labels)
    print(
        f"{split_name} loss: {loss:.4f}, {split_name} AUC: {auc:.4f}, "
        f"{split_name} Recall: {recall:.4f}, {split_name} F1: {f1_score:.4f}, {split_name} Acc: {accuracy:.4f}"
    )

In [ ]:
import json
import random
from pathlib import Path

import torch
from loader import load_data
from model import TripleMRNet
from submit_predictions import get_study

# 1) 模型路径（abnormal 已在前文给出，ACL / meniscus 按你的最佳 checkpoint 填写）
ABNORMAL_MODEL_PATH = BEST_ABNORMAL_MODEL_PATH
ACL_MODEL_PATH = "runs/acl_triple_alex/val0.1288_train0.0736_epoch23"
MENISCUS_MODEL_PATH = "runs/meniscus_triple_alex/val0.2679_train0.1832_epoch9"
BACKBONE = "alexnet"

# 2) 设备
use_gpu = torch.cuda.is_available()
device = torch.device("cuda" if use_gpu else "cpu")


def load_model(model_path):
    model = TripleMRNet(backbone=BACKBONE)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def predict_with_device(model, tensors, abnormality_prior=None):
    with torch.no_grad():
        vol_axial = tensors["axial"].to(device)
        vol_sagit = tensors["sagit"].to(device)
        vol_coron = tensors["coron"].to(device)

        logit = model.forward(vol_axial, vol_sagit, vol_coron)
        pred = torch.sigmoid(logit)
        pred_value = float(pred.detach().cpu().numpy()[0][0])

        if abnormality_prior is not None:
            pred_value = float(pred_value * abnormality_prior)
        return pred_value


# 3) 从 abnormal 验证集随机抽一例（保证 study_id 与三视图一致）
_, valid_loader = load_data("abnormal", use_gpu)
random_seed = 42
random.seed(random_seed)

sample_count = len(valid_loader.dataset.paths)
random_index = random.randrange(sample_count)
filename = valid_loader.dataset.paths[random_index]
study_id = Path(filename).stem

axial_path = str(Path("data/valid/axial") / filename)
sagittal_path = str(Path("data/valid/sagittal") / filename)
coronal_path = str(Path("data/valid/coronal") / filename)

study_tensors = get_study(axial_path, sagittal_path, coronal_path)

# 4) 三模型推理（ACL、meniscus 乘 abnormal prior，与 submit_predictions 一致）
abnormal_model = load_model(ABNORMAL_MODEL_PATH)
acl_model = load_model(ACL_MODEL_PATH)
meniscus_model = load_model(MENISCUS_MODEL_PATH)

p_abnormal = predict_with_device(abnormal_model, study_tensors)
p_acl = predict_with_device(acl_model, study_tensors, abnormality_prior=p_abnormal)
p_meniscus = predict_with_device(meniscus_model, study_tensors, abnormality_prior=p_abnormal)

print(f"study_id: {study_id}")
print(f"abnormal={p_abnormal:.4f}, acl={p_acl:.4f}, meniscus={p_meniscus:.4f}")

# 5) 调用 LLM 生成结构化报告
structured_report = explain_diagnosis_with_qwen(
    study_id=study_id,
    abnormal_prob=p_abnormal,
    acl_prob=p_acl,
    meniscus_prob=p_meniscus,
    model="qwen3-max",
)

print(json.dumps(structured_report, indent=2, ensure_ascii=False))

In [ ]:
!pip install pandas

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, recall_score, f1_score, confusion_matrix
)
# =========================
# 0) 项目路径
# =========================
PROJECT_ROOT = os.path.abspath("..")  # 仓库根目录（本笔记本在 notebooks/ 下）
os.chdir(PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from loader import load_data
from model import TripleMRNet
from evaluate import run_model
# =========================
# 1) 任务配置（按你自己的最佳模型改路径）
# =========================
TASK_CONFIGS = {
    "abnormal": {
        "model_path": "runs/abnormal_triple_alex/val0.1758_train0.0854_epoch24",
        "abnormal_model_path": None
    },
    "acl": {
        "model_path": "runs/acl_triple_alex/val0.1370_train0.0717_epoch22",
        # 若你评估 ACL 时要乘 abnormal prior，这里填 abnormal ckpt
        "abnormal_model_path": "runs/abnormal_triple_alex/val0.1758_train0.0854_epoch24"
    },
    "meniscus": {
        "model_path": "runs/meniscus_triple_alex/val0.2641_train0.1819_epoch10",
        # 同理
        "abnormal_model_path": "runs/abnormal_triple_alex/val0.1758_train0.0854_epoch24"
    },
}
BACKBONE = "alexnet"
USE_GPU = torch.cuda.is_available()
THRESHOLD = 0.5
# 输出目录
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
# =========================
# 2) 评估函数
# =========================
def evaluate_one_task(task_name, model_path, abnormal_model_path=None):
    _, valid_loader = load_data(task_name, USE_GPU)
    model = TripleMRNet(backbone=BACKBONE)
    state_dict = torch.load(model_path, map_location=(None if USE_GPU else "cpu"))
    model.load_state_dict(state_dict)
    if USE_GPU:
        model = model.cuda()
    val_loss, val_auc, preds, labels = run_model(
        model=model,
        loader=valid_loader,
        train=False,
        optimizer=None,
        abnormal_model_path=abnormal_model_path
    )
    preds = np.array(preds, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    pred_bin = (preds >= THRESHOLD).astype(np.int32)
    # 指标
    auc_score = float(roc_auc_score_safe(labels, preds))
    ap_score = float(average_precision_score(labels, preds))
    acc = float(accuracy_score(labels, pred_bin))
    rec = float(recall_score(labels, pred_bin, zero_division=0))
    f1 = float(f1_score(labels, pred_bin, zero_division=0))
    tn, fp, fn, tp = confusion_matrix(labels, pred_bin).ravel()
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    result = {
        "task": task_name,
        "valid_loss": float(val_loss),
        "AUC": auc_score,
        "PR_AUC(AP)": ap_score,
        "Accuracy": acc,
        "Recall(Sensitivity)": rec,
        "Specificity": specificity,
        "F1": f1,
        "n_samples": int(len(labels)),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
        "threshold": THRESHOLD
    }
    return labels, preds, result
def roc_auc_score_safe(y_true, y_score):
    # 防止极端情况下单类导致 roc_auc 崩溃
    if len(np.unique(y_true)) < 2:
        return np.nan
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(y_true, y_score)
# =========================
# 3) 画图函数
# =========================
def plot_and_save(task_name, labels, preds):
    # ROC
    fpr, tpr, _ = roc_curve(labels, preds)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.title(f"ROC Curve ({task_name})")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    roc_path = os.path.join(FIG_DIR, f"{task_name}_roc.png")
    plt.tight_layout()
    plt.savefig(roc_path, dpi=300)
    plt.close()
    # PR
    precision, recall, _ = precision_recall_curve(labels, preds)
    ap = average_precision_score(labels, preds)
    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, linewidth=2, label=f"AP = {ap:.4f}")
    plt.title(f"Precision-Recall Curve ({task_name})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend(loc="lower left")
    plt.grid(alpha=0.3)
    pr_path = os.path.join(FIG_DIR, f"{task_name}_pr.png")
    plt.tight_layout()
    plt.savefig(pr_path, dpi=300)
    plt.close()
    # 并排图（可用于论文）
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    axes[0].set_title(f"ROC ({task_name})")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].legend(loc="lower right")
    axes[0].grid(alpha=0.3)
    axes[1].plot(recall, precision, linewidth=2, label=f"AP = {ap:.4f}")
    axes[1].set_title(f"PR ({task_name})")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend(loc="lower left")
    axes[1].grid(alpha=0.3)
    combined_path = os.path.join(FIG_DIR, f"{task_name}_roc_pr_combined.png")
    fig.tight_layout()
    fig.savefig(combined_path, dpi=300)
    plt.close(fig)
    return roc_path, pr_path, combined_path
# =========================
# 4) 主流程
# =========================
all_rows = []
all_plot_paths = []
for task, cfg in TASK_CONFIGS.items():
    labels, preds, row = evaluate_one_task(
        task_name=task,
        model_path=cfg["model_path"],
        abnormal_model_path=cfg["abnormal_model_path"]
    )
    all_rows.append(row)
    roc_path, pr_path, combined_path = plot_and_save(task, labels, preds)
    all_plot_paths.append({
        "task": task,
        "roc_path": roc_path,
        "pr_path": pr_path,
        "combined_path": combined_path
    })
    print(f"[{task}] done | loss={row['valid_loss']:.4f}, AUC={row['AUC']:.4f}, AP={row['PR_AUC(AP)']:.4f}")
# 汇总表
df_summary = pd.DataFrame(all_rows).sort_values("task")
csv_path = os.path.join(FIG_DIR, "metrics_summary_3tasks.csv")
df_summary.to_csv(csv_path, index=False)
print("\n=== Summary ===")
print(df_summary.to_string(index=False))
print("\nSaved CSV:", csv_path)
print("\nSaved plots:")
for p in all_plot_paths:
    print(p)

In [ ]:
# ========= One-cell Error Analysis Pipeline =========
import os
import sys
import csv
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score, confusion_matrix

# -------------------------
# 0) 基础配置（按你环境改）
# -------------------------
PROJECT_ROOT = os.path.abspath("..")  # 仓库根目录（本笔记本在 notebooks/ 下）
os.chdir(PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

from loader import load_data
from model import TripleMRNet
from evaluate import run_model

USE_GPU = torch.cuda.is_available()
BACKBONE = "alexnet"
THRESHOLD = 0.5
TOP_K = 8
OUT_DIR = "error_analysis_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# 1) 模型路径配置（务必改成你真实存在的 ckpt）
# -------------------------
CKPT = {
    "abnormal": "runs/abnormal_triple_alex/val0.1758_train0.0854_epoch24",
    "acl": "runs/acl_triple_alex/val0.1370_train0.0717_epoch22",          # <-- 改
    "meniscus": "runs/meniscus_triple_alex/val0.2641_train0.1819_epoch10",  # <-- 改
}
ABN_FOR_PRIOR = CKPT["abnormal"]  # ACL/MENISCUS 乘先验时使用

# -------------------------
# 2) 工具函数
# -------------------------
def safe_auc(y_true, y_prob):
    y_true = np.array(y_true)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_prob))

def build_error_type(y, yp):
    if y == 1 and yp == 1:
        return "TP"
    if y == 0 and yp == 0:
        return "TN"
    if y == 0 and yp == 1:
        return "FP"
    return "FN"

def evaluate_cases(task, model_path, abnormal_model_path=None, threshold=0.5):
    _, valid_loader = load_data(task, USE_GPU)

    model = TripleMRNet(backbone=BACKBONE)
    state = torch.load(model_path, map_location=(None if USE_GPU else "cpu"))
    model.load_state_dict(state)
    if USE_GPU:
        model = model.cuda()

    val_loss, _, preds, labels = run_model(
        model=model,
        loader=valid_loader,
        train=False,
        optimizer=None,
        abnormal_model_path=abnormal_model_path
    )

    preds = np.array(preds, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    pred_bin = (preds >= threshold).astype(np.int32)

    study_ids = [p.split(".")[0] for p in valid_loader.dataset.paths]

    rows = []
    for sid, y, p, yp in zip(study_ids, labels, preds, pred_bin):
        rows.append({
            "task": task,
            "study_id": sid,
            "y_true": int(y),
            "y_prob": float(p),
            "y_pred": int(yp),
            "error_type": build_error_type(int(y), int(yp))
        })

    df = pd.DataFrame(rows)

    tn, fp, fn, tp = confusion_matrix(labels, pred_bin).ravel()
    summary = {
        "task": task,
        "valid_loss": float(val_loss),
        "AUC": safe_auc(labels, preds),
        "n_samples": int(len(labels)),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "threshold": float(threshold),
        "with_prior": bool(abnormal_model_path is not None)
    }
    return df, summary

def top_fp_fn(df, k=8):
    fp = df[df.error_type == "FP"].sort_values("y_prob", ascending=False).head(k)
    fn = df[df.error_type == "FN"].sort_values("y_prob", ascending=True).head(k)
    return fp, fn

def quality_features(study_id, datadir="data/valid"):
    out = {}
    for view in ["axial", "sagittal", "coronal"]:
        arr = np.load(str(Path(datadir) / view / f"{study_id}.npy"))  # [s,h,w]
        out[f"{view}_slices"] = int(arr.shape[0])
        out[f"{view}_range"] = float(arr.max() - arr.min())
        out[f"{view}_std"] = float(arr.std())
    return out

def attach_features(df, datadir="data/valid"):
    feats = []
    for sid in df.study_id.tolist():
        f = quality_features(sid, datadir=datadir)
        f["study_id"] = sid
        feats.append(f)
    feat_df = pd.DataFrame(feats)
    return df.merge(feat_df, on="study_id", how="left")

def compare_error_vs_correct(df_feat):
    tmp = df_feat.copy()
    tmp["is_error"] = tmp["error_type"].isin(["FP", "FN"]).astype(int)
    cols = [c for c in tmp.columns if c.endswith("_slices") or c.endswith("_range") or c.endswith("_std")]
    if not cols:
        return pd.DataFrame()
    return tmp.groupby("is_error")[cols].mean().T

# -------------------------
# 3) 主流程：三个任务主评估
# -------------------------
all_case_dfs = {}
all_summaries = []

for task in ["abnormal", "acl", "meniscus"]:
    abn_prior = None if task == "abnormal" else ABN_FOR_PRIOR
    df_task, sum_task = evaluate_cases(
        task=task,
        model_path=CKPT[task],
        abnormal_model_path=abn_prior,
        threshold=THRESHOLD
    )
    all_case_dfs[task] = df_task
    all_summaries.append(sum_task)

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(os.path.join(OUT_DIR, "summary_main.csv"), index=False)

print("\n=== Main Summary (with prior for ACL/Meniscus) ===")
print(summary_df.to_string(index=False))

# -------------------------
# 4) Top-K FP/FN + 特征对比
# -------------------------
for task in ["abnormal", "acl", "meniscus"]:
    df = all_case_dfs[task]
    fp, fn = top_fp_fn(df, k=TOP_K)

    fp_path = os.path.join(OUT_DIR, f"{task}_top_fp.csv")
    fn_path = os.path.join(OUT_DIR, f"{task}_top_fn.csv")
    fp.to_csv(fp_path, index=False)
    fn.to_csv(fn_path, index=False)

    print(f"\n=== {task.upper()} Top-{TOP_K} FP ===")
    print(fp[["study_id", "y_true", "y_prob", "y_pred", "error_type"]].to_string(index=False))

    print(f"\n=== {task.upper()} Top-{TOP_K} FN ===")
    print(fn[["study_id", "y_true", "y_prob", "y_pred", "error_type"]].to_string(index=False))

    # 错误组 vs 正确组质量特征
    df_feat = attach_features(df, datadir="data/valid")
    feat_cmp = compare_error_vs_correct(df_feat)
    feat_cmp_path = os.path.join(OUT_DIR, f"{task}_error_vs_correct_features.csv")
    feat_cmp.to_csv(feat_cmp_path)

    print(f"\n=== {task.upper()} Feature Mean: Correct(is_error=0) vs Error(is_error=1) ===")
    if feat_cmp.shape[0] > 0:
        print(feat_cmp.to_string())
    else:
        print("No feature columns found.")

# -------------------------
# 5) 先验影响分析（ACL / Meniscus）
# -------------------------
prior_compare_rows = []
for task in ["acl", "meniscus"]:
    # no prior
    df_no, sum_no = evaluate_cases(
        task=task, model_path=CKPT[task], abnormal_model_path=None, threshold=THRESHOLD
    )
    # with prior
    df_yes, sum_yes = evaluate_cases(
        task=task, model_path=CKPT[task], abnormal_model_path=ABN_FOR_PRIOR, threshold=THRESHOLD
    )

    prior_compare_rows.append({
        "task": task,
        "AUC_no_prior": sum_no["AUC"],
        "AUC_with_prior": sum_yes["AUC"],
        "FP_no_prior": sum_no["FP"],
        "FP_with_prior": sum_yes["FP"],
        "FN_no_prior": sum_no["FN"],
        "FN_with_prior": sum_yes["FN"],
        "delta_AUC": sum_yes["AUC"] - sum_no["AUC"] if pd.notna(sum_yes["AUC"]) and pd.notna(sum_no["AUC"]) else np.nan
    })

prior_df = pd.DataFrame(prior_compare_rows)
prior_df.to_csv(os.path.join(OUT_DIR, "prior_impact_acl_meniscus.csv"), index=False)

print("\n=== Prior Impact (ACL / Meniscus) ===")
print(prior_df.to_string(index=False))

print(f"\nAll outputs saved to: {OUT_DIR}")

In [ ]:
# ===== AlexNet-only Ablation: Prior + Threshold =====
import os
import sys
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, recall_score, f1_score, confusion_matrix
)

# -----------------------
# 0) 路径与导入
# -----------------------
PROJECT_ROOT = os.path.abspath("..")  # 仓库根目录（本笔记本在 notebooks/ 下）
os.chdir(PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

from loader import load_data
from model import TripleMRNet
from evaluate import run_model

# -----------------------
# 1) 仅 AlexNet checkpoint
# -----------------------
ABNORMAL_CKPT = "runs/abnormal_triple_alex/val0.1758_train0.0854_epoch24"  # 改
ACL_CKPT      = "runs/acl_triple_alex/val0.1370_train0.0717_epoch22"                  # 改
MENISCUS_CKPT = "runs/meniscus_triple_alex/val0.2641_train0.1819_epoch10"        # 改

USE_GPU = torch.cuda.is_available()
BACKBONE = "alexnet"

OUT_DIR = "ablation_outputs_alexnet"
os.makedirs(OUT_DIR, exist_ok=True)

THRESHOLDS = [0.3, 0.5, 0.7]

# -----------------------
# 2) 评估函数
# -----------------------
def evaluate_task(task, model_path, threshold=0.5, abnormal_model_path=None):
    _, valid_loader = load_data(task, USE_GPU)

    model = TripleMRNet(backbone=BACKBONE)
    state = torch.load(model_path, map_location=(None if USE_GPU else "cpu"))
    model.load_state_dict(state)
    if USE_GPU:
        model = model.cuda()

    val_loss, _, preds, labels = run_model(
        model=model,
        loader=valid_loader,
        train=False,
        optimizer=None,
        abnormal_model_path=abnormal_model_path
    )

    preds = np.array(preds, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    pred_bin = (preds >= threshold).astype(np.int32)

    auc = roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else np.nan
    ap = average_precision_score(labels, preds)
    acc = accuracy_score(labels, pred_bin)
    rec = recall_score(labels, pred_bin, zero_division=0)
    f1  = f1_score(labels, pred_bin, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, pred_bin).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "task": task,
        "threshold": threshold,
        "with_prior": bool(abnormal_model_path is not None),
        "valid_loss": float(val_loss),
        "AUC": float(auc),
        "PR_AUC": float(ap),
        "Accuracy": float(acc),
        "Recall": float(rec),
        "Specificity": float(spec),
        "F1": float(f1),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
        "n_samples": int(len(labels))
    }

# -----------------------
# 3) 实验矩阵
# -----------------------
rows = []

# 3.1 abnormal（无 prior）
for th in THRESHOLDS:
    rows.append(evaluate_task(
        task="abnormal",
        model_path=ABNORMAL_CKPT,
        threshold=th,
        abnormal_model_path=None
    ))

# 3.2 ACL（有/无 prior）
for use_prior in [False, True]:
    for th in THRESHOLDS:
        rows.append(evaluate_task(
            task="acl",
            model_path=ACL_CKPT,
            threshold=th,
            abnormal_model_path=(ABNORMAL_CKPT if use_prior else None)
        ))

# 3.3 meniscus（有/无 prior）
for use_prior in [False, True]:
    for th in THRESHOLDS:
        rows.append(evaluate_task(
            task="meniscus",
            model_path=MENISCUS_CKPT,
            threshold=th,
            abnormal_model_path=(ABNORMAL_CKPT if use_prior else None)
        ))

df = pd.DataFrame(rows)

# -----------------------
# 4) 输出结果
# -----------------------
# 主表（完整）
full_csv = os.path.join(OUT_DIR, "alexnet_ablation_full.csv")
df.to_csv(full_csv, index=False)

# 对照表：prior 影响（固定阈值0.5）
prior_table = df[(df["task"].isin(["acl", "meniscus"])) & (df["threshold"] == 0.5)].copy()
prior_csv = os.path.join(OUT_DIR, "alexnet_prior_ablation_threshold_0.5.csv")
prior_table.to_csv(prior_csv, index=False)

# 对照表：阈值影响（按任务展开）
thr_table = df[df["with_prior"] == False].copy()  # 可改成 True 看 prior 下阈值变化
thr_csv = os.path.join(OUT_DIR, "alexnet_threshold_ablation_no_prior.csv")
thr_table.to_csv(thr_csv, index=False)

print("=== AlexNet Ablation Summary (all settings) ===")
print(df.sort_values(["task", "with_prior", "threshold"]).to_string(index=False))

print("\nSaved files:")
print(full_csv)
print(prior_csv)
print(thr_csv)

# 额外打印：0.5阈值下最关键结果
print("\n=== Key results @ threshold=0.5 ===")
print(df[df["threshold"] == 0.5].sort_values(["task", "with_prior"]).to_string(index=False))